# Phase II: Phishing Email Detection

## Baseline (Logistic Regression + TF-IDF) vs Fine-Tuned DistilBERT

**Course:** PROG74040, Advanced Topics in Artificial Intelligence and Machine Learning
**Team:** Jonathan Taylor & Isaiah Andrews
**Task:** Binary text classification of emails (legitimate vs phishing)

This notebook records our Phase II work, following the structure of the Week 10 (`SW10Lab_Hugging_Face`) and Week 11 (`SW11Lab_Deployment`) labs:

1. **Setup:** we installed the Hugging Face libraries (as stated in SW10/SW11)
2. **Data:** loaded the Kaggle phishing email dataset (just as in Phase I)
3. **Baseline:** the application of logistic regression using TF-IDF (as in Phase I)
4. **Advanced:** a finely tuned version of `distilbert-base-uncased` using the Hugging Face Trainer API
5. **Comparison:** both models were evaluated using the same criteria
6. **Deployment:** we put the model on the Hugging Face Hub and made the Streamlit web demo available

## 1. Setup and Dependencies

We installed torch, transformers, datasets, evaluate, accelerate, kagglehub, and the plotting libraries we use.

In [ ]:
# Install from the official PyTorch repository so that torch, torchvision, and torchaudio all use the same CUDA build.
get_ipython().run_line_magic('pip', 'install -q --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio')
get_ipython().run_line_magic('pip', 'install -q --upgrade transformers datasets evaluate accelerate')
get_ipython().run_line_magic('pip', 'install -q "huggingface-hub>=1.2.0,<2.0" kagglehub scikit-learn pandas matplotlib seaborn')

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch, torchvision, torchaudio
import transformers
import accelerate

import kagglehub
from datasets import Dataset
from scipy.special import softmax

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             f1_score, precision_score, recall_score, accuracy_score)

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, pipeline,
                          TrainingArguments, Trainer)

from huggingface_hub import login, notebook_login, whoami, create_repo, upload_folder

print(f"PyTorch {torch.__version__} (CUDA available: {torch.cuda.is_available()})")


## 2. Loading the dataset

We used the same source as in Phase I: naserabdullahalam/phishing-email-dataset on Kaggle (~82k emails constructed from Enron, CEAS_08, SpamAssassin, Ling, Nazario, and Nigerian_Fraud).

In [ ]:
path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

# The Kaggle folder contains individual CSV files for the email collections (e.g., CEAS_08, Enron).
# The combined, deduplicated dataset with the text and label columns is named phishing_email.csv.
csv_file = os.path.join(path, "phishing_email.csv")
print("Downloaded to:", csv_file)

df = pd.read_csv(csv_file)
print(f"Dataset shape: {df.shape}")
df.info()

In [ ]:
print(df['label'].value_counts())
print(f"\nPhishing ratio: {df['label'].mean():.2%}")



## 3. Preprocessing and the splitting of the data into training, validation, and test sets

- We used the column named `text_combined` (the same one as in the Phase I baseline notebook).
- The data was divided in the ratio 80/10/10 so that the balance of the class would be maintained in each subset.

In [ ]:

X = df['text_combined'].astype(str)
y = df['label'].astype(int)

# 80% train, 10% validation, 10% test  (stratified on the label)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

print(f"Train:        {len(X_tr):>6,}  (phishing {y_tr.mean():.2%})")
print(f"Validation:   {len(X_val):>6,}  (phishing {y_val.mean():.2%})")
print(f"Test:         {len(X_test):>6,}  (phishing {y_test.mean():.2%})")


## 4. Baseline Model: Logistic Regression, TF-IDF

We started with a simple model that uses logistic regression and TF-IDF features. TF-IDF, which stands for term frequency-inverse document frequency, helps us turn text into numbers that show how important each word is in a document compared to others in the dataset. Logistic regression then uses these numbers to predict the outcome. This model gives us a basic point of comparison before we try more complex methods.

It is the baseline for Phase I and remains the point of reference required by the project rubric, since it converts each email into a **TF-IDF** vector and classifies it using **Logistic Regression**.

In [ ]:

vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=20000,
    stop_words='english',
    sublinear_tf=True,
    min_df=5,
    max_df=0.7,
)

X_tr_tfidf = vectorizer.fit_transform(X_tr)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)
print("TF-IDF matrix shape:", X_tr_tfidf.shape)

baseline = LogisticRegression(C=1.0, class_weight='balanced',
                              solver='liblinear', max_iter=1000, random_state=42)
baseline.fit(X_tr_tfidf, y_tr)
print("Baseline fitted.")


In [ ]:

def evaluate_model(y_true, y_prob, y_pred, name):
    print("=" * 60)
    print(f"  {name}")
    print("=" * 60)
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score : {f1_score(y_true, y_pred):.4f}")
    print(f"AUC-ROC  : {roc_auc_score(y_true, y_prob):.4f}")
    print(classification_report(y_true, y_pred, target_names=['Legitimate', 'Phishing']))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "auc_roc": roc_auc_score(y_true, y_prob),
    }

base_proba = baseline.predict_proba(X_test_tfidf)[:, 1]
base_pred  = baseline.predict(X_test_tfidf)
baseline_metrics = evaluate_model(y_test, base_proba, base_pred, "BASELINE: Logistic Regression + TF-IDF")

cm = confusion_matrix(y_test, base_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
plt.title('Baseline Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()

## 5. An Advanced Model: The Fine-Tuning of DistilBERT

### 5.1 Why DistilBERT?

We employed a smaller variant of BERT (about 40% smaller and 60% faster) while still maintaining approximately 97% of the original BERT's ability to understand language. It is therefore well-suited to serve as a fast, lightweight, fine-tuned model, since it can be trained quickly even on a CPU and runs smoothly on a free GPU (for example, the one provided by Kaggle).

### 5.2 What is behind `pipeline()`?

The Hugging Face `pipeline()` API hides three steps:
1. **Preprocessing:** converts the raw text into tokens, then those tokens are turned into token IDs, and finally, the token IDs are transformed into tensors.
2. **Model inference:** the tensors are transmitted through the Transformer to the classification head to generate the logits.
3. **Postprocessing:** converts the logits into softmax probabilities and then produces the labels and scores.

In [ ]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

### 5.3 Set up the Hugging Face dataset

We converted the pandas splits for the train / validation / test sets into tokenized `datasets.Dataset` objects.

In [ ]:
MAX_LEN = 256  # DistilBERT supports up to 512 tokens; emails fit well below that

def make_hf_dataset(Xs, ys):
    ds = Dataset.from_dict({"text": Xs.tolist(), "label": ys.tolist()})
    return (
        ds.map(
            lambda b: tokenizer(b["text"], padding="max_length", truncation=True,
                                max_length=MAX_LEN),
            batched=True,
        )
        .rename_column("label", "labels")
        .remove_columns(["text"])
        .with_format("torch")
    )

train_ds = make_hf_dataset(X_tr, y_tr)
val_ds   = make_hf_dataset(X_val, y_val)
test_ds  = make_hf_dataset(X_test, y_test)

print("Train:", train_ds, "\nVal:", val_ds, "\nTest:", test_ds)


In [ ]:
id2label = {0: "Legitimate", 1: "Phishing"}
label2id = {"Legitimate": 0, "Phishing": 1}

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


### 5.4 Training details

The hyperparameters we chose for a fast, reproducible run:

| Hyperparameter      | Value  | Notes |
|---------------------|--------|-------|
| Model               | `distilbert-base-uncased` | lightweight (~66M params) |
| Max sequence length | 256    | emails fit comfortably |
| Batch size          | 16     | safe for free GPU runtimes & CPU |
| Optimizer           | AdamW  | standard for transformers |
| Learning rate       | 2e-5   | typical for fine-tuning |
| Weight decay        | 0.01   | a small form of regularization |
| Epochs              | 2      | fast, good baseline result |

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }

training_args = TrainingArguments(
    output_dir="./distilbert-phishing",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


### 5.5 Train (fine-tune) DistilBERT

Training on the free GPU that Kaggle provides required only a few minutes, and with each epoch, the Trainer displayed the loss together with the validation set metrics.

In [ ]:
trainer.train()


### 5.6 Evaluate using the held-out test set

To make a fair side-by-side comparison, we used the same metrics as in the baseline.

In [ ]:
preds_out = trainer.predict(test_ds)
test_logits = preds_out.predictions
test_labels = preds_out.label_ids
test_preds  = np.argmax(test_logits, axis=-1)
test_proba  = softmax(test_logits, axis=-1)[:, 1]

transformer_metrics = evaluate_model(test_labels, test_proba, test_preds,
                                     "ADVANCED: Fine-tuned DistilBERT")

cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
plt.title('Fine-tuned DistilBERT Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()


## 6. Results: Baseline vs Fine-Tuned DistilBERT

The table and chart below compare the two models on the same test set.

In [ ]:
results = pd.DataFrame([baseline_metrics, transformer_metrics],
                        index=["Baseline (LR+TF-IDF)", "Fine-tuned DistilBERT"])
results.index.name = "Model"
results.round(4)


In [ ]:
results.T.plot(kind="bar", figsize=(10, 5), rot=0,
              title="Baseline vs Fine-Tuned DistilBERT: test set metrics")
plt.ylabel("Score"); plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


In [ ]:
# Quick sanity check on hand-written example emails
examples = [
    "Dear user, your account has been compromised. Click here to verify your details immediately.",
    "Hi team, please find attached the Q3 report for your review.",
    "CONGRATULATIONS! You have won a free iPhone. Claim your prize now at this link.",
    "Thank you for your inquiry, we have processed your request.",
]
preds = trainer.predict(make_hf_dataset(pd.Series(examples), pd.Series([0, 0, 1, 0])))
for text, p in zip(examples, softmax(preds.predictions, axis=-1)):
    lab = "Phishing" if p[1] > p[0] else "Legitimate"
    print(f"[{lab:>9}] ({p[1]:.0%})  {text[:70]}...")


## 7. Deployment: the Hugging Face Hub and the Streamlit Community Cloud

We uploaded the fine-tuned model to the Hugging Face Model Hub (which is free; the same hub that was used in SW10/SW11). While the GitHub repository contains only the app code, the Streamlit app downloads the model's weights directly from the Hugging Face Hub whenever it runs. For the interactive demonstration, we selected Streamlit Community Cloud rather than using a Gradio Space because new Gradio Spaces on Hugging Face require a paid PRO subscription. Streamlit is free, can be deployed directly from this public GitHub repository, and is listed as an acceptable deployment option in the project rubric (as well as in the SW11 references).

In [ ]:
# We save the fine-tuned model and tokenizer to a local folder first, so we can push them to the Hub.


In [ ]:
# To publish to the Hugging Face Hub, we log in with a WRITE access token that
# we store as a Kaggle secret named HF_TOKEN.


In [ ]:

HF_USERNAME     = whoami()['name']         # filled automatically from your logged-in HF account
MODEL_REPO_NAME = "phishing-email-distilbert"
MODEL_REPO_ID   = f"{HF_USERNAME}/{MODEL_REPO_NAME}"

create_repo(repo_id=MODEL_REPO_ID, repo_type="model", exist_ok=True)
upload_folder(folder_path=LOCAL_MODEL_DIR, repo_id=MODEL_REPO_ID, repo_type="model")
print(f"Model uploaded to: https://huggingface.co/{MODEL_REPO_ID}")

In [ ]:
# Verify the model loads straight from the Hugging Face Hub (same call the deployed app uses)
hub_classifier = pipeline("text-classification", model=MODEL_REPO_ID, tokenizer=MODEL_REPO_ID)
hub_classifier("Congratulations! You have won a free iPhone. Click here to claim your prize.")


### 7.1 Prepare the Streamlit app files

The Streamlit Community Cloud app runs directly from the files in this GitHub repository, which is why we included a small `streamlit_app/` folder containing `app.py` and `requirements.txt`. It retrieves the fine-tuned model from the Hugging Face Hub via `pipeline()` and provides a simple interface that lets users paste text for classification.

In [ ]:
STREAMLIT_DIR = "streamlit_app"
os.makedirs(STREAMLIT_DIR, exist_ok=True)

app_py = f"""import streamlit as st
from transformers import pipeline

st.set_page_config(page_title="Phishing Email Detector", layout="centered")

@st.cache_resource
def load_classifier():
    # Load the fine-tuned DistilBERT model straight from the Hugging Face Hub
    return pipeline("text-classification", model="{MODEL_REPO_ID}")

classifier = load_classifier()

st.title("Phishing Email Detector")
st.write("Fine-tuned DistilBERT that flags phishing emails. Trained on the Kaggle phishing-email dataset.")

email = st.text_area("Paste an email here:", height=200)

if st.button("Analyze") and email.strip():
    result = classifier(email)[0]
    label = result["label"]
    score = result["score"]
    if label == "Legitimate":
        st.success(f"Legitimate ({score:.1%} confidence)")
    else:
        st.error(f"Phishing ({score:.1%} confidence)")
    st.markdown("---")
    st.caption("Model: fine-tuned distilbert-base-uncased | Metric: label + confidence")
"""

with open(os.path.join(STREAMLIT_DIR, "app.py"), "w") as f:
    f.write(app_py)

with open(os.path.join(STREAMLIT_DIR, "requirements.txt"), "w") as f:
    f.write("streamlit\ntransformers\ntorch\n")

print("Streamlit app files ready in", STREAMLIT_DIR)
print(os.listdir(STREAMLIT_DIR))

### 7.2 Deploy to the Streamlit Community Cloud (free)

With the app files in the repo, deployment was a one-time manual step in the browser, and the only part we did not automate:

1. Added the `streamlit_app/` folder to this GitHub repository.
2. Went to share.streamlit.io and logged in with our GitHub account.
3. Clicked the **Create app** option, picked the repository, set the main file path to `streamlit_app/app.py`, and selected `main` as the branch.
4. Obtained a public URL such as `https://<appname>.streamlit.app` by clicking **Deploy**.

Since the repository is public, the app can retrieve the model from the Hugging Face Hub without a token, allowing users to open the link and enter any email address to get a Legitimate/Phishing verdict.

## 8. Summary and Limitations

- **Baseline (LR + TF-IDF):** it is strong, quick, and interpretable, which is why it serves as an excellent reference point.
- **Fine-tuned DistilBERT:** captures word order and context in a way that TF-IDF cannot, and it outperforms the baseline on all the metrics (with an accuracy of 0.9918 compared to 0.9841).
- **Deployment:** the model can be obtained from the Hugging Face Hub, and the interactive demo is provided completely free of charge through the Streamlit Community Cloud, so anyone who has the link can use it.

**Limitations of the deployed system:**
- Since DistilBERT is smaller than full BERT, there is still a possibility that some edge cases could get through.
- The biases of the model come from the data it was trained on; it can miss new or heavily obfuscated phishing attempts (for example, text that has been encoded or entirely new URLs).
- Emails with more than 256 subwords are truncated (we could increase `MAX_LEN` to allow longer bodies).
- The free Streamlit tier causes the application to sleep after a period of inactivity, and the first user has to wake it up, which makes the first load take a long time.